In [1]:
import json
import os
import pickle as pkl

import numpy as np
from sentence_transformers import SentenceTransformer

#### Objective:

Using `sentence-transformers/all-mpnet-base-v2`, generates embeddings for all the crawled vendors.

#### Depends on:

In [ ]:
vendor_to_idx_map_fname = "./data/norm_vendors_list_to_idx_map.pkl"
scrap_files_dir = "data/scrap_output"

Note that inside `scrap_files_dir`, the notebook expects to find files generated by the python crawler found here: 

https://github.com/cassinaooo/python-vendor-crawler

For example, if the normalized vendor name results in "OFFICE MAX", it expects a file named `office_max` inside `scrap_files_dir`, with the multiple lines following this structure:
```

```json
{"company_url": "https://www.officemax.com/","company_text": "send on your partner an by store credit media our inc all rights","company_name": "data/out/scrap_output/office_max"}
```

#### Generates:

In [1]:
vendor_to_vector_fname = "./data/vendor_to_vector.pkl"

----------------------

In [2]:
with open(vendor_to_idx_map_fname, "rb") as f:
    norm_vendors_str = pkl.load(f)

In [3]:
norm_vendors_str[:5]

['A & N',
 'A&B ECO SAFE PEST CONT',
 'A&B HOME GROUP',
 'A&C FIRE EXTINQUISHER',
 'A&D SUPPLY']

In [4]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

In [5]:
vendor_to_vector = {}

In [6]:
for i, v in enumerate(norm_vendors_str):
    f_name = v.replace(" ", "_").lower()

    scrap_file_location = f"{scrap_files_dir}/{f_name}"

    if os.path.exists(scrap_file_location) and os.path.getsize(scrap_file_location) > 0:
        print("existing file. creating vendor vector")
    else:
        continue

    print(scrap_file_location)
    print(v)
    print(i, "/", len(norm_vendors_str))

    with open(scrap_file_location, "r") as f:
        vendor_to_vector[v] = {"text": []}
        for line in f.readlines():
            parsed_json_line = json.loads(line)
            vendor_to_vector[v]["text"].append(parsed_json_line["company_text"])

        embeddings = model.encode(vendor_to_vector[v]["text"])
        vendor_to_vector[v]["vectors"] = embeddings
        vendor_to_vector[v]["mean"] = np.mean(embeddings, axis=0)

print("done")

existing file. creating vendor vector
data/scrap_output/agent_fee
AGENT FEE
792 / 30650
existing file. creating vendor vector
data/scrap_output/airgas_central
AIRGAS CENTRAL
915 / 30650
existing file. creating vendor vector
data/scrap_output/amazon
AMAZON
1288 / 30650
existing file. creating vendor vector
data/scrap_output/american_ai
AMERICAN AI
1382 / 30650
existing file. creating vendor vector
data/scrap_output/apl_apple_online_store
APL APPLE ONLINE STORE
2035 / 30650
existing file. creating vendor vector
data/scrap_output/at&t
AT&T
2581 / 30650
existing file. creating vendor vector
data/scrap_output/att_bus_phone_pmt
ATT BUS PHONE PMT
2674 / 30650
existing file. creating vendor vector
data/scrap_output/att_nbi
ATT NBI
2676 / 30650
existing file. creating vendor vector
data/scrap_output/atw_stillwater
ATW STILLWATER
2703 / 30650
existing file. creating vendor vector
data/scrap_output/best_buy
BEST BUY
3424 / 30650
existing file. creating vendor vector
data/scrap_output/bill_warren_

In [7]:
with open(vendor_to_vector_fname, "wb") as f:
    pkl.dump(
        vendor_to_vector,
        f,
    )